# BÀI THỰC HÀNH 4 — SO SÁNH ĐỘ TƯƠNG ĐỒNG ẢNH BẰNG WAVELET (WAVELET HASH)

**Mục tiêu:**
- Dùng biến đổi Wavelet (thư viện PyWavelets) để trích xuất đặc trưng ảnh.
- Biến đặc trưng thành **mã băm nhị phân**, so sánh bằng **khoảng cách Hamming**.
- Đánh giá bằng Accuracy, Sensitivity, Specificity và đường cong ROC.

**Ảnh đầu vào:** đặt 8 file `cat_1.png, cat_2.png, dog_1.png, dog_2.png, car_1.png, car_2.png, dog_3.png, dog_4.png` cùng thư mục với notebook.

**Kết quả:** biểu đồ ROC + bảng thông số được lưu thành `ket_qua.png`.

**Ý tưởng chung:** ảnh giống nhau → mã băm giống nhau → khoảng cách Hamming nhỏ.

## 0. Thư viện

- `pywt`: biến đổi Wavelet.
- `PIL`: đọc ảnh, đổi sang xám, đổi kích thước.
- `sklearn.metrics`: tính accuracy, ma trận nhầm lẫn, ROC, AUC.

In [ ]:
import pywt
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc

## 0.1. Tạo ảnh (`cat_2.png` và `car_2.png`)

- Chỉnh sửa `cat_1.png` (xoay nhẹ, làm mờ, thêm nhiễu hạt) và lưu thành `cat_2.png`.
- Đổi màu đỏ trên `car_1.png` sang xanh dương và lưu thành `car_2.png`.

In [ ]:
# --- Phần 1: Chỉnh cat_1.png -> lưu thành cat_2.png ---
img = Image.open("cat_1.png")

# 1. Xoay góc lệch nhẹ
img = img.rotate(6, resample=Image.Resampling.BILINEAR)

# 2. Làm hơi mờ
img = img.filter(ImageFilter.GaussianBlur(radius=1.0))

# 3. Thêm hạt nhẹ
arr = np.asarray(img, dtype=np.float32)
noise = np.random.normal(0.0, 12.0, size=arr.shape)
img = Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

img.save("cat_2.png")


# --- Phần 2: Đổi màu đỏ trên car_1.png -> lưu thành car_2.png ---
car = Image.open("car_1.png").convert("RGB")
car_arr = np.asarray(car, dtype=np.float32)

r, g, b = car_arr[:, :, 0], car_arr[:, :, 1], car_arr[:, :, 2]

# Nhận diện vùng màu đỏ (R cao hơn rõ rệt so với G và B)
red_mask = (r > 110) & ((r - g) > 45) & ((r - b) > 45)

# Đổi sang màu xanh dương (giữ lại độ sáng tương đối)
new_arr = car_arr.copy()
new_arr[red_mask, 0] = b[red_mask] * 0.4
new_arr[red_mask, 1] = g[red_mask] * 0.8 + 40
new_arr[red_mask, 2] = np.clip(r[red_mask] * 1.1, 0, 255)

Image.fromarray(np.clip(new_arr, 0, 255).astype(np.uint8)).save("car_2.png")

## 1. Trích xuất đặc trưng Wavelet và tạo mã băm 

Các bước trong hàm `wavelet_hash`:

1. Đọc ảnh, chuyển sang **ảnh xám** (`convert("L")`) và đổi về **64×64** để mọi ảnh có cùng kích thước → mã băm cùng độ dài.
2. Biến đổi **Wavelet rời rạc 2D** `wavedec2` với wavelet `db4`, 3 mức phân rã. Kết quả phần tử `[0]` là **cA** (hệ số xấp xỉ, chứa hình dạng tổng thể, đã loại chi tiết nhỏ và nhiễu). Với `db4` và ảnh 64×64, cA có kích thước 14×14 = **196 hệ số**.
3. **Lượng tử hóa:** so từng hệ số với **trung vị** của cA: lớn hơn → `1`, ngược lại → `0`. Kết quả là mã băm nhị phân 196 bit.

Dùng trung vị nên mã băm chia đều 0/1 và không phụ thuộc độ sáng tuyệt đối của ảnh.

In [ ]:
# 1. Tạo mã băm Wavelet cho ảnh
def wavelet_hash(img_path, wavelet="db4", level=3):
    img = Image.open(img_path).convert("L").resize((64, 64))
    cA = pywt.wavedec2(np.array(img), wavelet, level=level)[0]
    return (cA.flatten() > np.median(cA)).astype(int)

## 2. Khoảng cách Hamming 

Khoảng cách Hamming = **tỷ lệ bit khác nhau** giữa hai mã băm, nằm trong [0, 1]:
- `0`: hai mã băm giống hệt nhau.
- Càng gần `1`: hai ảnh càng khác nhau (xấp xỉ 0.5 với hai ảnh không liên quan).

In [ ]:
# 2. Tính khoảng cách Hamming
def hamming_distance(h1, h2):
    return np.mean(h1 != h2)

## 3. Chuẩn bị dữ liệu 

Mỗi ảnh đi kèm một **nhãn thực tế**. Hai ảnh được coi là **tương tự** nếu cùng nhãn.

Trong danh sách này, `dog_2.png` được gán nhãn `horse` nên khác nhãn với các ảnh `dog` còn lại.
Do đó: 8 ảnh → 28 cặp, trong đó **5 cặp tương tự** (cat_1–cat_2, car_1–car_2, và 3 cặp trong nhóm dog_1, dog_3, dog_4) và **23 cặp không tương tự**.

Sau đó tạo mã băm cho toàn bộ ảnh.

In [ ]:
# 3. Danh sách ảnh và nhãn thực tế
data = [
    ("cat_1.png", "cat"), ("cat_2.png", "cat"),
    ("dog_1.png", "dog"), ("dog_2.png", "horse"),
    ("car_1.png", "car"), ("car_2.png", "car"),
    ("dog_3.png", "dog"), ("dog_4.png", "dog"),
]

hashes = [wavelet_hash(path) for path, _ in data]
labels = [label for _, label in data]

print("Số ảnh:", len(data), "| Độ dài mã băm:", len(hashes[0]), "bit")

## 4. So sánh từng cặp ảnh

Duyệt mọi cặp `(i, j)` với `i < j` (mỗi cặp chỉ xét một lần):
- `distances`: khoảng cách Hamming của cặp.
- `y_true`: nhãn đúng, `1` nếu cùng nhãn (tương tự), `0` nếu khác nhãn.

In [ ]:
# 4. So sánh từng cặp ảnh
y_true, distances = [], []

for i in range(len(data)):
    for j in range(i + 1, len(data)):
        distances.append(hamming_distance(hashes[i], hashes[j]))
        y_true.append(1 if labels[i] == labels[j] else 0)

y_true = np.array(y_true)
distances = np.array(distances)

print("Tổng số cặp:", len(y_true), "| Cặp tương tự:", y_true.sum(), "| Cặp không tương tự:", (y_true == 0).sum())

## 5. Phân loại và đánh giá 

**Quy tắc phân loại:** khoảng cách Hamming `≤ threshold (0.35)` → dự đoán *tương tự* (1), ngược lại *không tương tự* (0).

**Ma trận nhầm lẫn:**

| | Dự đoán tương tự | Dự đoán không tương tự |
|---|---|---|
| **Thực tế tương tự** | TP | FN |
| **Thực tế không tương tự** | FP | TN |

**Các chỉ số:**
- **Accuracy (độ chính xác)** = (TP + TN) / tổng số cặp → tỷ lệ cặp phân loại đúng.
- **Sensitivity (độ nhạy)** = TP / (TP + FN) → trong các cặp thực sự tương tự, bao nhiêu cặp được nhận ra.
- **Specificity (độ đặc hiệu)** = TN / (TN + FP) → trong các cặp thực sự khác nhau, bao nhiêu cặp được nhận ra.

**ROC:** thay đổi ngưỡng liên tục để vẽ TPR theo FPR. Vì khoảng cách nhỏ nghĩa là giống, ta dùng `scores = 1 - distances` (điểm càng cao càng giống) để đưa vào `roc_curve`. **AUC** càng gần 1 thì thuật toán càng tốt; AUC = 0.5 tương đương đoán ngẫu nhiên.

In [ ]:
# 5. Phân loại và tính các chỉ số
threshold = 0.35
y_pred = (distances <= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

accuracy = accuracy_score(y_true, y_pred)
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

scores = 1 - distances
fpr, tpr, _ = roc_curve(y_true, scores)
roc_auc = auc(fpr, tpr)

print(f"Accuracy    : {accuracy:.2%}")
print(f"Sensitivity : {sensitivity:.2%}")
print(f"Specificity : {specificity:.2%}")
print(f"AUC         : {roc_auc:.3f}")
print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")

## 6. Vẽ đường cong ROC và bảng thông số

Một hình duy nhất gồm đường ROC (xanh), đường chéo ngẫu nhiên (nét đứt) và khung ghi các chỉ số. Hình được lưu thành `ket_qua.png`.

In [ ]:
# 6. Hiển thị biểu đồ và thông số chung một hình
fig, ax = plt.subplots(figsize=(9, 6))

# Vẽ ROC
ax.plot(
    fpr, tpr,
    color="blue",
    linewidth=2,
    label=f"Wavelet Hash (AUC = {roc_auc:.3f})"
)

ax.plot([0, 1], [0, 1], "k--", label="Random Classifier")

ax.set_xlabel("FPR (1 - Specificity)", fontsize=11)
ax.set_ylabel("TPR (Sensitivity)", fontsize=11)
ax.set_title("ROC Curve - Wavelet Hash", fontsize=14, fontweight="bold")

ax.legend(loc="lower right")
ax.grid(alpha=0.3)


# Khung hiển thị thông số
metrics_text = (
    "EVALUATION METRICS\n"
    "---------------------------\n"
    f"Accuracy       : {accuracy:.2%}\n"
    f"Sensitivity    : {sensitivity:.2%}\n"
    f"Specificity    : {specificity:.2%}\n"
    f"AUC            : {roc_auc:.3f}\n"
    "---------------------------\n"
    f"Threshold      : {threshold}\n"
    f"TP             : {tp}\n"
    f"TN             : {tn}\n"
    f"FP             : {fp}\n"
    f"FN             : {fn}"
)

ax.text(
    0.05, 0.95,
    metrics_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round,pad=0.6",
        facecolor="white",
        edgecolor="gray",
        alpha=0.9
    ),
    family="monospace"
)

plt.tight_layout()

# Lưu ảnh kết quả
plt.savefig("ket_qua.png", dpi=300, bbox_inches="tight")

plt.show()

print("Đã lưu kết quả tại: ket_qua.png")

## 7.Nhận xét kết quả 

| Chỉ số / Thành phần | Giá trị | Nhận xét |
| :--- | :--- | :--- |
| **Accuracy** | `78.57%` | Đạt mức khá trên bộ dữ liệu gồm **8 mẫu ảnh (4 cặp đối chiếu)**. |
| **Sensitivity (TPR)** | `60.00%` | Khả năng phát hiện đúng các cặp ảnh tương tự ở mức trung bình — nhận diện được phần lớn cặp giống nhau nhưng vẫn bỏ sót trường hợp cùng đối tượng nhưng khác góc chụp, ánh sáng hoặc phông nền. |
| **Specificity** | `82.61%` | Khả năng phân biệt đúng các cặp ảnh khác loại tốt, ít xảy ra tình trạng báo động giả (nhận nhầm ảnh khác loại thành giống nhau). |
| **AUC** | `0.661` | Mức phân tách trung bình (cao hơn đường ngẫu nhiên `0.50`). Đường ROC vọt lên mức `TPR = 0.60` ngay tại `FPR = 0.00` (nhận diện tốt các cặp giống nhau rõ rệt), sau đó đi ngang kéo dài ở đoạn giữa nên diện tích dưới đường cong chưa cao. |
| **Threshold** | `0.35` | Mức ngưỡng hiện tại thiên về sự an toàn, giúp giữ độ đặc hiệu (`Specificity`) cao và hạn chế ghép nhầm ảnh khác loại. |
| **TP / TN / FP / FN** | `TP = 3`, `TN = 19`<br>`FP = 4`, `FN = 2` | Tỷ lệ dự đoán đúng chiếm đa số (`TP + TN`), tuy nhiên do quy mô dữ liệu chỉ có **8 mẫu (4 cặp)** nên đường ROC có dạng bậc thang rõ rệt và chỉ số dễ biến động khi thay đổi mẫu. |

**Tóm lại:**
- **Ưu điểm:** Wavelet Hash giữ được cấu trúc tổng thể của ảnh và khử nhiễu chi tiết nhỏ tốt, giúp nhận diện chính xác ngay các cặp ảnh có bố cục tương đồng rõ rệt (`TPR = 0.60` khi `FPR = 0`) và phân biệt ảnh khác loại khá ổn định (`Specificity = 82.61%`).
- **Hạn chế:** Nhạy cảm với sự thay đổi góc nhìn, xoay/lật hoặc phông nền khác nhau giữa hai ảnh cùng loại; đồng thời số lượng mẫu thử nghiệm còn ít (**8 mẫu / 4 cặp**) nên kết quả mang tính minh họa và cần thêm dữ liệu để đánh giá toàn diện hơn.